# COMP47470 Assignment 4: Apache Spark GraphFrames and Streaming

This assignment has problems on Apache Spark GraphFrames and Streaming.

The assignment carries 10 points (10% weight).

There are two sets of questions. The first set of questions is on GraphFrame operations. The second set is on Structured Streaming.

The input datasets are given in a ZIP file, **datasets.zip**. Unzip the file before proceeding to solve the assignment.

## Component1: Apache Spark GraphFrame API

For this component, you must use the following two input datasets.

Data in the following dataset represents a station where users can pickup or return bikes.

**data/bike-data/201508_station_data.csv**

The dataset has following record format.

*station_id, name, lat, long, dockcount, landmark, installation*

**You must use the name field as the vertex ID.**

Data about individual bike trips is in the following dataset.

**data/bike-data/201508_trip_data.csv**

The dataset has following record format.

*Trip ID, Duration, Start Date, Start Station, Start Terminal, End Date, End Station, End Terminal, Bike #, Subscriber Type, Zip Code*

**You must use the Start Station and End Station fields as src and dst to represent an edge.**

In [1]:
# Installing Java and Spark:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [2]:
#Switching java version to use as default (choose option 2)
!update-alternatives --config java
#Switching javac version to use as default (choose option 2)
!update-alternatives --config javac
#Switching jps version to use as default (choose option 2)
!update-alternatives --config jps

There are 2 choices for the alternative java (providing /usr/bin/java).

  Selection    Path                                            Priority   Status
------------------------------------------------------------
* 0            /usr/lib/jvm/java-17-openjdk-amd64/bin/java      1711      auto mode
  1            /usr/lib/jvm/java-17-openjdk-amd64/bin/java      1711      manual mode
  2            /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java   1081      manual mode

Press <enter> to keep the current choice[*], or type selection number: 2
update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java to provide /usr/bin/java (java) in manual mode
There are 2 choices for the alternative javac (providing /usr/bin/javac).

  Selection    Path                                          Priority   Status
------------------------------------------------------------
* 0            /usr/lib/jvm/java-17-openjdk-amd64/bin/javac   1711      auto mode
  1            /usr/lib/jvm/java-17-o

In [4]:
!wget https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!rm spark-3.5.1-bin-hadoop3.tgz


--2025-11-15 15:42:48--  https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400446614 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.1-bin-hadoop3.tgz’

spark-3.5.1-bin-had 100%[===================>] 381.90M  1.99MB/s    in 4m 23s  

2025-11-15 15:47:12 (1.45 MB/s) - ‘spark-3.5.1-bin-hadoop3.tgz’ saved [400446614/400446614]



In [5]:
#Checking Java default version
!java -version

openjdk version "1.8.0_462"
OpenJDK Runtime Environment (build 1.8.0_462-8u462-ga~us1-0ubuntu2~22.04.2-b08)
OpenJDK 64-Bit Server VM (build 25.462-b08, mixed mode)


In [6]:
# Setting up our environmental variables:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

In [7]:
!pip install -q findspark
import findspark
findspark.init()

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import desc
from pyspark import *

In [9]:
# Configuring GraphFrames:
conf = SparkConf()
conf.set("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.5-s_2.12")
conf.set("spark.sql.repl.eagerEval.enabled", True)

# Starting our Spark Session:
spark = SparkSession.builder.master("local[*]").config(conf=conf).getOrCreate()
spark

In [10]:
from graphframes import *

In [11]:
from pyspark.sql import SparkSession
from graphframes import GraphFrame
from pyspark.sql.functions import col

In [12]:

from google.colab import files
uploaded = files.upload()


Saving 201508_station_data.csv to 201508_station_data.csv
Saving 201508_trip_data.csv to 201508_trip_data.csv


In [13]:

stations_df = spark.read.csv("/content/201508_station_data.csv", header=True, inferSchema=True)
trips_df = spark.read.csv("/content/201508_trip_data.csv", header=True, inferSchema=True)


In [14]:
from graphframes import GraphFrame
from pyspark.sql.functions import col

# Rename 'name' to 'id' for vertices
vertices = stations_df.withColumnRenamed("name", "id")

# Prepare edges with src, dst, and Duration as integer
edges = trips_df.select(
    col("Start Station").alias("src"),
    col("End Station").alias("dst"),
    col("Duration").cast("int")
)


In [15]:
# Create GraphFrame
g = GraphFrame(vertices, edges)


In [17]:
# First 5 rows of stations
stations_df.show(5)

# First 5 rows of trips
trips_df.show(5)


+----------+--------------------+---------+-----------+---------+--------+------------+
|station_id|                name|      lat|       long|dockcount|landmark|installation|
+----------+--------------------+---------+-----------+---------+--------+------------+
|         2|San Jose Diridon ...|37.329732|-121.901782|       27|San Jose|    8/6/2013|
|         3|San Jose Civic Ce...|37.330698|-121.888979|       15|San Jose|    8/5/2013|
|         4|Santa Clara at Al...|37.333988|-121.894902|       11|San Jose|    8/6/2013|
|         5|    Adobe on Almaden|37.331415|  -121.8932|       19|San Jose|    8/5/2013|
|         6|    San Pedro Square|37.336721|-121.894074|       15|San Jose|    8/7/2013|
+----------+--------------------+---------+-----------+---------+--------+------------+
only showing top 5 rows

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+
|Trip ID|Duration|     Start D

### Problem1: Write the GraphFrame API operation(s) that outputs the **longest** bike ride, which is the ride that took the **maximum** duration from a source station to a destination station?

Use the **Duration** field to query for the duration of the ride. Note you must convert the string to an integer.

You can use either subgraphs employing filters or motif DSL to answer this problem.

**What are the GraphFrame API operations and the output from their execution?**

In [16]:
longest_ride = edges.orderBy(col("Duration").desc()).limit(1)
longest_ride.show()


+--------------------+-------------+--------+
|                 src|          dst|Duration|
+--------------------+-------------+--------+
|South Van Ness at...|2nd at Folsom|17270400|
+--------------------+-------------+--------+



### Problem2: Write the GraphFrame API operation(s) that outputs the bike station (**id**) with the **minimum** triangle count (**not equal to 0**)?

**What are the GraphFrame API operations and the output from their execution?**

In [18]:
triangle_counts = g.triangleCount()
min_triangle_station = triangle_counts.filter(col("count") > 0).orderBy("count").limit(1)
min_triangle_station.select("id", "count").show()


+--------------------+-----+
|                  id|count|
+--------------------+-----+
|Redwood City Medi...|   14|
+--------------------+-----+



### Problem3: Write a Spark GraphFrame query that finds and outputs the bike station (**id**) that received the **most/maximum** bike traffic using the GraphFrame PageRank algorithm?

The query must employ the following algorithmic parameters:

```python
resetProbability=0.15
maxIter=100
```

**What are the GraphFrame API operations and the output from their execution?**

In [19]:
pagerank_results = g.pageRank(resetProbability=0.15, maxIter=100)
top_station = pagerank_results.vertices.orderBy(col("pagerank").desc()).limit(1)
top_station.select("id", "pagerank").show()


+--------------------+-----------------+
|                  id|         pagerank|
+--------------------+-----------------+
|San Jose Diridon ...|4.052074047679068|
+--------------------+-----------------+



### Problem4: You will write a triangle motif in GraphFrame DSL where the starting and ending station is '**San Jose Diridon Caltrain Station**'.

### Write Spark GraphFrame queries that process rides that form a triangle pattern between three stations (**a**,**b**,**c**). The following motif represents the triangle pattern:

### "(a)-[ab]->(b); (b)-[bc]->(c); (c)-[ca]->(a)"

### The output of the GraphFrame API operations must be the number of rides by the same bike (**Bike \#**) between stations (**a**,**b**,**c**) where the sum of the ride times between **a and b** and **b and c** is **equal** to the ride time between **c and a**.

Note **a** must represent the station with name, **San Jose Diridon Caltrain Station**.

In addition, **a**, **b**, and **c** must represent stations with different names in the station dataset (vertices with different IDs).

You **must** use **only** the following constraints on dates in the query:

1). The start date of the ride from **a** to **b** must be less than the start date from **b** to **c**.

2). The start date of the ride from **b** to **c** must be less than the start date from **c** to **a**.

You must convert the 'Start Date' and 'End Date' fields to Spark timestamps.

Use the Duration field to query for the duration of the ride. However, you must convert the string Duration to an integer.

**What are the GraphFrame API operations and the output from their execution?**


In [25]:
# Set LEGACY parser to handle date format in Spark 3+
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

from pyspark.sql.functions import col, to_timestamp
from graphframes import GraphFrame

# vertices
vertices = stations_df.withColumnRenamed("name", "id")  # 'id' is required for GraphFrames

# edges with Duration and timestamps
edges = trips_df.select(
    col("Start Station").alias("src"),
    col("End Station").alias("dst"),
    col("Duration").cast("int"),
    to_timestamp(col("Start Date"), "M/d/yyyy H:mm").alias("start_ts"),
    to_timestamp(col("End Date"), "M/d/yyyy H:mm").alias("end_ts"),
    col("Bike #")
)

# Create the GraphFrame
g = GraphFrame(vertices, edges)

# Find triangle motifs a -> b -> c -> a
motifs = g.find("(a)-[ab]->(b); (b)-[bc]->(c); (c)-[ca]->(a)")

In [26]:
# Apply constraints:
# - a is 'San Jose Diridon Caltrain Station'
# - a, b, c are distinct
# - Dates increase: ab.start_ts < bc.start_ts < ca.start_ts
# - Duration condition: ab + bc == ca
triangles = motifs.filter(
    (col("a.id") == "San Jose Diridon Caltrain Station") &
    (col("a.id") != col("b.id")) &
    (col("a.id") != col("c.id")) &
    (col("b.id") != col("c.id")) &
    (col("ab.start_ts") < col("bc.start_ts")) &
    (col("bc.start_ts") < col("ca.start_ts")) &
    ((col("ab.Duration") + col("bc.Duration")) == col("ca.Duration"))
)

In [27]:
# Count rides by the same bike
triangles_by_bike = triangles.groupBy("ab.`Bike #`").count()
triangles_by_bike.show()

+------+-----+
|Bike #|count|
+------+-----+
|   148|  195|
|   243|  229|
|    31|  843|
|   251|  144|
|    65|  314|
|    53|   11|
|   133|  174|
|   296|  283|
|    78|   70|
|   155|  195|
|   108|  133|
|   683|   48|
|   211|  128|
|   126|  210|
|   101|   86|
|    81|  317|
|    28|  346|
|   210|   48|
|   183|    3|
|   688|   21|
+------+-----+
only showing top 20 rows



## Component2: Apache Spark's Structured Streaming API

For this component, you must use the Human Activity Recognition data, which is a stream of files arriving one at a time in the following directory:

**data/activity-data**

Each file contains smart watch sensor data in JSON format containing the following fields:

```python
Timestamps (Arrival_Time, Creation_Time)
Smartphone and smartwatch models
User and device information
```

In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count


In [31]:
from google.colab import files
uploaded = files.upload()  # Select activity-data.zip


Saving activity-data.zip to activity-data.zip


In [32]:
!unzip activity-data.zip -d /content/


Archive:  activity-data.zip
   creating: /content/activity-data/
  inflating: /content/activity-data/part-00000-tid-730451297822678341-1dda7027-2071-4d73-a0e2-7fb6a91e1d1f-0-c000.json  
  inflating: /content/activity-data/part-00001-tid-730451297822678341-1dda7027-2071-4d73-a0e2-7fb6a91e1d1f-0-c000.json  
  inflating: /content/activity-data/part-00002-tid-730451297822678341-1dda7027-2071-4d73-a0e2-7fb6a91e1d1f-0-c000.json  
  inflating: /content/activity-data/part-00003-tid-730451297822678341-1dda7027-2071-4d73-a0e2-7fb6a91e1d1f-0-c000.json  
  inflating: /content/activity-data/part-00004-tid-730451297822678341-1dda7027-2071-4d73-a0e2-7fb6a91e1d1f-0-c000.json  
  inflating: /content/activity-data/part-00005-tid-730451297822678341-1dda7027-2071-4d73-a0e2-7fb6a91e1d1f-0-c000.json  
  inflating: /content/activity-data/part-00006-tid-730451297822678341-1dda7027-2071-4d73-a0e2-7fb6a91e1d1f-0-c000.json  
  inflating: /content/activity-data/part-00007-tid-730451297822678341-1dda7027-2071-4d73

### Problem1: Write a Spark Structured Streaming query that outputs **count** of events where the user is **sitting**.

The **gt** field is **sit** for the sitting activity.

You must provide the following trigger when reading the stream, **maxFilesPerTrigger** equal to 1.

The query will write the results into an in-memory table, **activity_counts** to store the counts.

You will use **complete** output mode.

You will use the following two functions to completely process the stream data (which is finite in this case). Assuming your streaming query is represented by the variable, **activityQuery**:

```python
    activityQuery.processAllAvailable()
    activityQuery.stop()
```

Finally, you will use the following **spark.sql** query to display the result table, **activity_counts**.

```python
spark.sql("SELECT * FROM activity_counts").show()
```

**What are the Spark Structured Streaming operations and the output from the above spark.sql query?**

In [36]:
from pyspark.sql.functions import col

# schema from existing files
staticDF = spark.read.json("/content/activity-data")
activity_schema = staticDF.schema

# read the folder as a streaming source using schema
activityStream = spark.readStream \
    .schema(activity_schema) \
    .option("maxFilesPerTrigger", 1) \
    .json("/content/activity-data")

# Filter for sitting activity and count
sittingCounts = activityStream.filter(col("gt") == "sit") \
    .groupBy() \
    .count()  # total sitting events

# Write results to an in-memory table
activityQuery = sittingCounts.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("activity_counts") \
    .start()

# Process all files and stop
activityQuery.processAllAvailable()
activityQuery.stop()

# Display the result table
spark.sql("SELECT * FROM activity_counts").show()


+------+
| count|
+------+
|984714|
+------+



### Problem2: Write a Spark Structured Streaming query that employs **event time stateful processing**, uses a **tumbling window** of length **10 minutes**, and outputs **count** of events where the user **g** is on the **bike**.

The **gt** field is **bike** and **User** is **g**.

You must provide the following trigger when reading the stream, **maxFilesPerTrigger** equal to 1.

You must use **Creation_Time** column for event time processing. Note the **Creation_Time** column must be converted to Spark SQL timestamp.

The query will write the results into an in-memory table, **events_per_tumbler** to store the counts.

You will use **complete** output mode.

You will use the following two functions to completely process the stream data (which is finite in this case). Assuming your streaming query is represented by the variable, **activityQuery**:

```python
    activityQuery.processAllAvailable()
    activityQuery.stop()
```

Finally, you will use the following spark.sql query to select the counts from the result table, **events_per_tumbler**.

```python
spark.sql("SELECT * FROM events_per_tumbler").show()
```

Use the following conf setting to avoid some runtime aborts/exceptions in tasks:

```python
spark.conf.set("spark.sql.shuffle.partitions", 16)
```

**What are the Spark Structured Streaming operations and the output from the above spark.sql query?**

In [39]:
from pyspark.sql.functions import col, window

spark.conf.set("spark.sql.shuffle.partitions", 16)

# Read stream with schema
staticDF = spark.read.json("/content/activity-data")
activity_schema = staticDF.schema

activityStream = spark.readStream \
    .schema(activity_schema) \
    .option("maxFilesPerTrigger", 1) \
    .json("/content/activity-data")

# Convert Creation_Time to timestamp
activityStream = activityStream.withColumn(
    "event_time",
    col("Creation_Time").cast("timestamp")
)

# Filter for User 'g' and gt 'bike'
bikeEvents = activityStream.filter(
    (col("User") == "g") & (col("gt") == "bike")
)

# 10-minute tumbling window
eventsPerTumbler = bikeEvents.groupBy(
    window(col("event_time"), "10 minutes")
).count()

# Write to in-memory table
activityQuery = eventsPerTumbler.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("events_per_tumbler") \
    .start()

# Process all files and stop
activityQuery.processAllAvailable()
activityQuery.stop()

# Display results
spark.sql("SELECT * FROM events_per_tumbler").show(truncate=False)


+------------------------------------------------+-----+
|window                                          |count|
+------------------------------------------------+-----+
|{-270696-01-03 10:00:00, -270696-01-03 10:10:00}|1    |
|{-211420-12-31 21:00:00, -211420-12-31 21:10:00}|1    |
|{-212726-05-31 18:30:00, -212726-05-31 18:40:00}|1    |
|{-210245-05-08 21:50:00, -210245-05-08 22:00:00}|1    |
|{-204912-03-06 00:10:00, -204912-03-06 00:20:00}|1    |
|{-267471-09-08 07:50:00, -267471-09-08 08:00:00}|1    |
|{-263947-06-22 12:00:00, -263947-06-22 12:10:00}|1    |
|{-210154-08-17 23:20:00, -210154-08-17 23:30:00}|1    |
|{-270355-03-27 13:40:00, -270355-03-27 13:50:00}|1    |
|{-264610-05-14 08:40:00, -264610-05-14 08:50:00}|1    |
|{-267027-04-02 23:20:00, -267027-04-02 23:30:00}|1    |
|{-213205-12-23 13:10:00, -213205-12-23 13:20:00}|1    |
|{-207018-05-06 16:40:00, -207018-05-06 16:50:00}|1    |
|{-206656-03-22 05:30:00, -206656-03-22 05:40:00}|1    |
|{-209208-11-09 20:10:00, -2092

### Problem3: Write a Spark Structured Streaming query that employs **event time stateful processing**, uses a **hopping window** of length 10 minutes and slide interval (hop size) equal to 5 minutes, and outputs **count** of events where the User **a** is using the stairs.

The **gt** field is **stairsup** or **stairsdown** and **User** is **a**.

You must provide the following trigger when reading the stream, **maxFilesPerTrigger** equal to 1.

You must use **Creation_Time** column for event time processing. Note the **Creation_Time** column must be converted to Spark SQL timestamp.

The query will write the results into an in-memory table, **events_per_hopper** to store the counts.

You will use **complete** output mode.

You will use the following two functions to completely process the stream data (which is finite in this case). Assuming your streaming query is represented by the variable, **activityQuery**:

```python
    activityQuery.processAllAvailable()
    activityQuery.stop()
```

You will use the following spark.sql query to select the counts from the result table, **events_per_hopper**.

```python
spark.sql("SELECT * FROM events_per_hopper").show()
```

Use the following conf setting to avoid some runtime aborts/exceptions in tasks:

```python
spark.conf.set("spark.sql.shuffle.partitions", 16)
```

**What are the Spark Structured Streaming operations and the output from the above spark.sql query?**

In [40]:
from pyspark.sql.functions import col, window

spark.conf.set("spark.sql.shuffle.partitions", 16)

# Read stream with schema
staticDF = spark.read.json("/content/activity-data")
activity_schema = staticDF.schema

activityStream = spark.readStream \
    .schema(activity_schema) \
    .option("maxFilesPerTrigger", 1) \
    .json("/content/activity-data")

# convert Creation_Time to timestamp
activityStream = activityStream.withColumn(
    "event_time",
    col("Creation_Time").cast("timestamp")
)

# Filter for User 'a' and stairs activities
stairsEvents = activityStream.filter(
    (col("User") == "a") & ((col("gt") == "stairsup") | (col("gt") == "stairsdown"))
)

# Aggregate counts using 10-minute hopping window with 5-minute slide
eventsPerHopper = stairsEvents.groupBy(
    window(col("event_time"), "10 minutes", "5 minutes")
).count()

# Write results to in-memory table
activityQuery = eventsPerHopper.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("events_per_hopper") \
    .start()

# Process all available files and stop the stream
activityQuery.processAllAvailable()
activityQuery.stop()

# Display the results
spark.sql("SELECT * FROM events_per_hopper").show(truncate=False)


+----------------------------------------------+-----+
|window                                        |count|
+----------------------------------------------+-----+
|{+84325-08-21 02:55:00, +84325-08-21 03:05:00}|1    |
|{+19472-08-09 06:05:00, +19472-08-09 06:15:00}|1    |
|{+83745-11-25 17:00:00, +83745-11-25 17:10:00}|1    |
|{+82151-12-06 06:05:00, +82151-12-06 06:15:00}|1    |
|{+11130-04-22 11:35:00, +11130-04-22 11:45:00}|1    |
|{+18334-12-14 08:15:00, +18334-12-14 08:25:00}|1    |
|{+15031-05-17 19:20:00, +15031-05-17 19:30:00}|1    |
|{+22667-11-20 23:10:00, +22667-11-20 23:20:00}|1    |
|{+78696-09-29 17:45:00, +78696-09-29 17:55:00}|1    |
|{+72509-03-13 06:10:00, +72509-03-13 06:20:00}|1    |
|{+72501-09-12 10:40:00, +72501-09-12 10:50:00}|1    |
|{+23334-01-08 00:00:00, +23334-01-08 00:10:00}|1    |
|{+17062-08-15 15:50:00, +17062-08-15 16:00:00}|1    |
|{+15763-06-27 11:35:00, +15763-06-27 11:45:00}|1    |
|{+82621-01-25 04:05:00, +82621-01-25 04:15:00}|1    |
|{+71084-0